## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [3]:
df = pl.read_csv("data/data.csv")

df

OrderNumber,SalesChannel,WarehouseCode,ProcuredDate,OrderDate,ShipDate,DeliveryDate,CurrencyCode,SalesTeamID,CustomerID,StoreID,ProductID,OrderQuantity,DiscountApplied,UnitCost,UnitPrice
str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,f64,str,str
"""SO - 000101""","""In-Store""","""WARE-UHY1004""","""31/12/17""","""31/5/18""","""14/6/18""","""19/6/18""","""USD""",6,15,259,12,5,0.08,"""$1,001.18""","""$1,963.10"""
"""SO - 000102""","""Online""","""WARE-NMK1003""","""31/12/17""","""31/5/18""","""22/6/18""","""2/7/2018""","""USD""",14,20,196,27,3,0.08,"""$3,348.66""","""$3,939.60"""
"""SO - 000103""","""Distributor""","""WARE-UHY1004""","""31/12/17""","""31/5/18""","""21/6/18""","""1/7/2018""","""USD""",21,16,213,16,1,0.05,"""$781.22""","""$1,775.50"""
"""SO - 000104""","""Wholesale""","""WARE-NMK1003""","""31/12/17""","""31/5/18""","""2/6/2018""","""7/6/2018""","""USD""",28,48,107,23,8,0.08,"""$1,464.69""","""$2,324.90"""
"""SO - 000105""","""Distributor""","""WARE-NMK1003""","""10/4/2018""","""31/5/18""","""16/6/18""","""26/6/18""","""USD""",22,49,111,26,8,0.1,"""$1,476.14""","""$1,822.40"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SO - 0008087""","""In-Store""","""WARE-MKL1006""","""26/9/20""","""30/12/20""","""7/1/2021""","""14/1/21""","""USD""",9,41,339,29,1,0.08,"""$121.94""","""$234.50"""
"""SO - 0008088""","""Online""","""WARE-NMK1003""","""26/9/20""","""30/12/20""","""2/1/2021""","""4/1/2021""","""USD""",14,29,202,3,6,0.05,"""$1,921.56""","""$3,202.60"""
"""SO - 0008089""","""Online""","""WARE-UHY1004""","""26/9/20""","""30/12/20""","""23/1/21""","""26/1/21""","""USD""",14,32,241,35,5,0.2,"""$2,792.76""","""$3,825.70"""


### Retrieve Basic Information About DataFrame

In [4]:
print(df.shape)
print(df.dtypes)

(7991, 16)
[String, String, String, String, String, String, String, String, Int64, Int64, Int64, Int64, Int64, Float64, String, String]


### Display Summary Statistics for All Columns

In [5]:
summary = df.describe()
print(summary)

shape: (9, 17)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ OrderNumb ┆ SalesChan ┆ Warehouse ┆ … ┆ OrderQuan ┆ DiscountA ┆ UnitCost  ┆ UnitPric │
│ ---       ┆ er        ┆ nel       ┆ Code      ┆   ┆ tity      ┆ pplied    ┆ ---       ┆ e        │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ str       ┆ ---      │
│           ┆ str       ┆ str       ┆ str       ┆   ┆ f64       ┆ f64       ┆           ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 7991      ┆ 7991      ┆ 7991      ┆ … ┆ 7991.0    ┆ 7991.0    ┆ 7991      ┆ 7991     │
│ null_coun ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 0.0       ┆ 0.0       ┆ 0         ┆ 0        │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ mean      ┆ null      ┆ null      ┆ null      ┆ … ┆ 4.525341  ┆ 0.115649  

### Find Longest Text Length in Each Column

In [6]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

OrderNumber,SalesChannel,WarehouseCode,ProcuredDate,OrderDate,ShipDate,DeliveryDate,CurrencyCode,UnitCost,UnitPrice
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
12,11,12,9,10,10,10,3,9,9


### Retrieve Unique Values for Select Features

In [7]:
# pl.Config.set_tbl_max_rows(None)
pl.Config.set_tbl_rows(1000)

cols_unique_vals = [
    "ProcuredDate",
    "OrderDate",
    "ShipDate",
    "DeliveryDate"
]

for col in cols_unique_vals:
    unique_vals = df.select(col).unique()
    print(f"{col}: {unique_vals}")

ProcuredDate: shape: (11, 1)
┌──────────────┐
│ ProcuredDate │
│ ---          │
│ str          │
╞══════════════╡
│ 27/10/18     │
│ 26/9/20      │
│ 10/4/2018    │
│ 19/7/18      │
│ 10/3/2020    │
│ 23/8/19      │
│ 1/12/2019    │
│ 18/6/20      │
│ 15/5/19      │
│ 31/12/17     │
│ 4/2/2019     │
└──────────────┘
OrderDate: shape: (945, 1)
┌────────────┐
│ OrderDate  │
│ ---        │
│ str        │
╞════════════╡
│ 21/11/18   │
│ 06/08/2020 │
│ 14/5/20    │
│ 02/05/2019 │
│ 05/05/2019 │
│ 27/8/19    │
│ 10/11/2018 │
│ 30/9/19    │
│ 13/12/19   │
│ 11/12/2019 │
│ 11/11/2018 │
│ 02/06/2020 │
│ 07/04/2019 │
│ 18/6/18    │
│ 15/8/19    │
│ 10/05/2020 │
│ 26/8/20    │
│ 24/5/20    │
│ 21/9/19    │
│ 25/11/19   │
│ 12/02/2019 │
│ 03/09/2019 │
│ 29/4/20    │
│ 27/5/19    │
│ 16/7/20    │
│ 14/6/18    │
│ 24/11/19   │
│ 05/01/2019 │
│ 05/07/2019 │
│ 05/10/2018 │
│ 30/8/19    │
│ 19/9/18    │
│ 20/5/20    │
│ 02/09/2019 │
│ 20/12/19   │
│ 07/11/2020 │
│ 30/12/20   │
│ 07/05/2019 │
│ 21/6/20 

### Retrieve Data Types of All Columns

In [8]:
print("Column data types:\n", df.dtypes)

Column data types:
 [String, String, String, String, String, String, String, String, Int64, Int64, Int64, Int64, Int64, Float64, String, String]


### Count Unique Values in Each Column

In [9]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                  Unique values in OrderNumber : 7991  
                 Unique values in SalesChannel : 4     
                Unique values in WarehouseCode : 6     
                 Unique values in ProcuredDate : 11    
                    Unique values in OrderDate : 945   
                     Unique values in ShipDate : 966   
                 Unique values in DeliveryDate : 966   
                 Unique values in CurrencyCode : 1     
                  Unique values in SalesTeamID : 28    
                   Unique values in CustomerID : 50    
                      Unique values in StoreID : 367   
                    Unique values in ProductID : 47    
                Unique values in OrderQuantity : 8     
              Unique values in DiscountApplied : 7     
                     Unique values in UnitCost : 5252  
                    Unique values in UnitPrice : 664   


### Check Distribution of Numerical Columns

In [10]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['id']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

SalesTeamID
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ SalesTeamID │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 7991.0      │
│ null_count ┆ 0.0         │
│ mean       ┆ 14.384307   │
│ std        ┆ 7.986086    │
│ min        ┆ 1.0         │
│ 25%        ┆ 8.0         │
│ 50%        ┆ 14.0        │
│ 75%        ┆ 21.0        │
│ max        ┆ 28.0        │
└────────────┴─────────────┘ 


CustomerID
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ CustomerID │
│ ---        ┆ ---        │
│ str        ┆ f64        │
╞════════════╪════════════╡
│ count      ┆ 7991.0     │
│ null_count ┆ 0.0        │
│ mean       ┆ 25.457014  │
│ std        ┆ 14.414883  │
│ min        ┆ 1.0        │
│ 25%        ┆ 13.0       │
│ 50%        ┆ 25.0       │
│ 75%        ┆ 38.0       │
│ max        ┆ 50.0       │
└────────────┴────────────┘ 


StoreID
shape: (9, 2)
┌────────────┬────────────┐
│ statistic  ┆ StoreID    │
│ ---     

### Retrieve Count of Nulls In Each Feature

In [11]:
def count_nulls(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "feature": df.columns,
        "null_count": df.null_count().row(0)
    })

pl.Config.set_tbl_rows(35)

null_counts = count_nulls(df)

pl.Config.set_tbl_rows(100)
null_counts

feature,null_count
str,i64
"""OrderNumber""",0
"""SalesChannel""",0
"""WarehouseCode""",0
"""ProcuredDate""",0
"""OrderDate""",0
"""ShipDate""",0
"""DeliveryDate""",0
"""CurrencyCode""",0
"""SalesTeamID""",0


### Display Unique Values For Each Feature

In [12]:
def list_unique_values_under_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count < threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
pl.Config.set_tbl_rows(1000)
list_unique_values_under_threshold(df)

Column: SalesChannel (4 unique values)
shape: (4,)
Series: 'SalesChannel' [str]
[
	"Distributor"
	"In-Store"
	"Online"
	"Wholesale"
]
--------------------------------------------------
Column: WarehouseCode (6 unique values)
shape: (6,)
Series: 'WarehouseCode' [str]
[
	"WARE-MKL1006"
	"WARE-NBV1002"
	"WARE-NMK1003"
	"WARE-PUJ1005"
	"WARE-UHY1004"
	"WARE-XYS1001"
]
--------------------------------------------------
Column: ProcuredDate (11 unique values)
shape: (11,)
Series: 'ProcuredDate' [str]
[
	"1/12/2019"
	"10/3/2020"
	"10/4/2018"
	"15/5/19"
	"18/6/20"
	"19/7/18"
	"23/8/19"
	"26/9/20"
	"27/10/18"
	"31/12/17"
	"4/2/2019"
]
--------------------------------------------------
Column: OrderDate (945 unique values)
shape: (945,)
Series: 'OrderDate' [str]
[
	"01/01/2019"
	"01/01/2020"
	"01/02/2019"
	"01/02/2020"
	"01/03/2019"
	"01/03/2020"
	"01/04/2019"
	"01/04/2020"
	"01/05/2019"
	"01/05/2020"
	"01/06/2018"
	"01/06/2019"
	"01/06/2020"
	"01/07/2018"
	"01/07/2019"
	"01/07/2020"
	"01/08/201